### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [ ]:
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])
print("Siap untuk latihan. Folder:", folder_tugas)

Siap untuk latihan. Folder: tugas10_stream


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [ ]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load("tugas10_stream")

df_hitung = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_stream = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query dimulai! Status:", query.status)

Streaming query dimulai! Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [ ]:
df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load("tugas10_stream")

agregasi_window = df_stream2.groupBy(window(col("waktu_pesan"), "2 minutes")).agg(
    spark_sum(col("qty") * col("harga_satuan")).alias("total_harga")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query dimulai! Status:", query2.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [ ]:
from datetime import datetime
waktu_sekarang = datetime.now()

# Batch 1
pd.DataFrame({
    "pesanan_id": ["P001"],
    "nama_cabang": ["Magelang"],
    "nama_menu": ["Americano"],
    "qty": [67],
    "harga_satuan": [15000],
    "waktu_pesan" : [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")],
}).to_csv(f"{'tugas10_stream'}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

pd.DataFrame({
    "pesanan_id": ["P002"],
    "nama_cabang": ["Jogja"],
    "nama_menu": ["Roti Bakar"],
    "qty": [50],
    "harga_satuan": [10000],
    "waktu_pesan" : [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")],
}).to_csv(f"{'tugas10_stream'}/batch_2.csv", index=False)

# Batch 2
print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P003"],
    "nama_cabang": ["Semarang"],
    "nama_menu": ["Nasi Goreng Spesial"],
    "qty": [21],
    "harga_satuan": [25000],
    "waktu_pesan" : [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")],
}).to_csv(f"{'tugas10_stream'}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+-----------+-----------+
|nama_cabang|total_harga|
+-----------+-----------+
|   Magelang|    1005000|
+-----------+-----------+

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+-----------+
|nama_cabang|total_harga|
+-----------+-----------+
|   Magelang|    1005000|
|      Jogja|     500000|
+-----------+-----------+

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+-----------+
|nama_cabang|total_harga|
+-----------+-----------+
|   Magelang|    1005000|
|   Semarang|     525000|
|      Jogja|     500000|
+-----------+-----------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.


In [ ]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

pd.DataFrame({
    "pesanan_id": ["P004"],
    "nama_cabang": ["Semarang"],
    "nama_menu": ["V60"],
    "qty": [56],
    "harga_satuan": [20000],
    "waktu_pesan" : [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")],
}).to_csv(f"{'tugas10_stream'}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark memproses...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_harga DESC").show()

Batch 4 dikirim. Menunggu Spark memproses...
+-----------+-----------+
|nama_cabang|total_harga|
+-----------+-----------+
|   Semarang|    1645000|
|   Magelang|    1005000|
|      Jogja|     500000|
+-----------+-----------+



In [ ]:
!ls tugas10_output

part-00000-cdedad04-33b0-40be-b75b-32ae0b304c4b-c000.snappy.parquet
part-00001-67227225-1df3-4a77-99ee-6316a2612012-c000.snappy.parquet
part-00002-036b86f3-263f-4c8d-aaf3-c05d5aa0149d-c000.snappy.parquet
part-00003-6f08e9f1-44c3-4cc1-a3bf-82bc85f1e550-c000.snappy.parquet
_spark_metadata


In [ ]:
df_hasil = spark.read.parquet("tugas10_output")
df_hasil.show(truncate=False)
print("Jumlah baris:", df_hasil.count())

+----------+-----------+-------------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu          |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------------+---+------------+-------------------+
|P003      |Semarang   |Nasi Goreng Spesial|21 |25000       |2026-11-05 08:53:15|
|P003      |Semarang   |Nasi Goreng Spesial|21 |25000       |2026-11-05 08:53:15|
|P001      |Magelang   |Americano          |67 |15000       |2026-11-05 08:53:15|
|P002      |Jogja      |Roti Bakar         |50 |10000       |2026-11-05 08:53:15|
+----------+-----------+-------------------+---+------------+-------------------+

Jumlah baris: 4


In [ ]:
query.stop()
query2.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
